In [1]:
# Script to sort data

import numpy as np
import pandas as pd
import sys

/tmp/ipykernel_463887/1210232043.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
# 1. Upload data

DATA = np.load('data/data_3DChannel_topo.pkl', allow_pickle=True)

# 2. Sort data by coil geometry

DATA_sorted = DATA.sort_values(by='geom')


In [3]:
DATA

,geom,src_x,src_y,rec_x,rec_y,midpx,op,ip
0,H2,-13.997982,-10.0,-11.997982,-10.0,-12.997982,0.002443,0.000495
1,H4,-13.997982,-10.0,-9.998168,-10.0,-11.997982,0.013184,0.003677
2,H8,-13.997982,-10.0,-5.997982,-10.0,-9.997982,0.052064,0.025379
3,V2,-13.997982,-10.0,-11.997982,-10.0,-12.997982,0.001465,0.000248
4,V4,-13.997982,-10.0,-9.998168,-10.0,-11.997982,0.008339,0.001784
...,...,...,...,...,...,...,...,...
4,V4,45.000808,49.0,49.001199,49.0,47.000808,0.114731,0.061773
5,V8,45.000808,49.0,53.000808,49.0,49.000808,0.178750,0.104775
6,P2,45.000808,49.0,47.100808,49.0,46.050808,0.002545,0.000263
7,P4,45.000808,49.0,49.100808,49.0,47.050808,0.014403,0.002003


In [7]:
len(DATA)/9

3600.0

In [12]:
# 3. Define number of position

n_coil_geom = 9 # Number of coil geometries

npos = int(len(DATA_sorted)/n_coil_geom) # number of positions

# number of data parameters per position

n_data_param = 18

# We have 3 additional points to the left of the first midpoint (assuming 
# 1 position per meter)
DATA_sorted_source = np.zeros((40*40, n_data_param))

# We will have positions with insufficient data in the edges

In [13]:
pos=0

for j in range(-1,39):
    for i in range(-1,39):
        dat = np.hstack((DATA_sorted.loc[(DATA_sorted.src_x > i+.06) & (DATA_sorted.src_x <=i +1.06) & 
                         (DATA_sorted.src_y > j+.06) & (DATA_sorted.src_y <= j+1.06)]['op'],
                         DATA_sorted.loc[(DATA_sorted.src_x > i+.06) & (DATA_sorted.src_x <=i +1.06) & 
                         (DATA_sorted.src_y > j+.06) & (DATA_sorted.src_y <= j+1.06)]['ip'])) 
      
        if dat.size == 0:
            continue
        if dat.size == 18:
            DATA_sorted_source[pos,:] = dat
            pos += 1
            print('position: ', pos)
            print('location: ')
            print(DATA_sorted.loc[(DATA_sorted.src_x > i+.06) & (DATA_sorted.src_x <=i +1.06) & 
                         (DATA_sorted.src_y > j+.06) & (DATA_sorted.src_y <= j+1.06)])
        print()

position:  1
location: 
  geom     src_x  src_y     rec_x  rec_y     midpx        op        ip
0   H2  0.006697    0.0  2.006697    0.0  1.006697  0.002433  0.000490
1   H4  0.006697    0.0  4.008192    0.0  2.006697  0.014054  0.003692
2   H8  0.006697    0.0  8.006697    0.0  4.006697  0.055509  0.025492
6   P2  0.006697    0.0  2.106697    0.0  1.056697  0.001160  0.000075
7   P4  0.006697    0.0  4.106697    0.0  2.056697  0.008932  0.000936
8   P8  0.006697    0.0  8.106697    0.0  4.056697  0.062461  0.011317
3   V2  0.006697    0.0  2.006697    0.0  1.006697  0.001430  0.000274
4   V4  0.006697    0.0  4.008192    0.0  2.006697  0.009058  0.002163
5   V8  0.006697    0.0  8.006697    0.0  4.006697  0.047055  0.015911

position:  2
location: 
  geom    src_x  src_y     rec_x  rec_y    midpx        op        ip
0   H2  1.00657    0.0  3.006570    0.0  2.00657  0.002475  0.000492
1   H4  1.00657    0.0  5.008655    0.0  3.00657  0.014460  0.003714
2   H8  1.00657    0.0  9.006570  

In [15]:
np.shape(DATA_sorted_source)

(1600, 18)

In [16]:
np.save('data/data_3DChannel_nosort_topo', DATA_sorted_source)